# ClinicalRAG — Query Demo

End-to-end walkthrough: retrieval, answer generation, and hallucination detection.

In [ ]:
import sys
import os
import json
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# add src to path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

print('imports ok')

In [ ]:
# --- dummy retriever (no GPU / API key needed) ---
import numpy as np

class DummyIndex:
    """Tiny in-memory cosine-similarity index for demo purposes."""

    PASSAGES = [
        {
            "pmid": "PMC3538468",
            "text": "Metformin is generally well tolerated. Common side effects include gastrointestinal "
                    "symptoms such as nausea, vomiting, diarrhoea, and abdominal pain, especially during "
                    "the initial weeks of treatment.",
            "section": "Adverse Effects",
        },
        {
            "pmid": "PMC6520897",
            "text": "Lactic acidosis is a rare but serious adverse effect associated with metformin, "
                    "occurring primarily in patients with renal impairment or conditions predisposing "
                    "to hypoxia.",
            "section": "Serious Adverse Events",
        },
        {
            "pmid": "PMC7246199",
            "text": "Long-term metformin use is associated with vitamin B12 deficiency in approximately "
                    "10-30% of patients, which may manifest as peripheral neuropathy or megaloblastic anaemia.",
            "section": "Nutritional Interactions",
        },
        {
            "pmid": "PMC4673959",
            "text": "Metformin does not cause hypoglycaemia when used as monotherapy because it does not "
                    "stimulate insulin secretion. It lowers hepatic glucose production via AMPK activation.",
            "section": "Mechanism of Action",
        },
        {
            "pmid": "PMC5860445",
            "text": "Metformin is contraindicated in patients with eGFR < 30 mL/min/1.73 m² due to "
                    "increased risk of metformin accumulation and lactic acidosis.",
            "section": "Contraindications",
        },
    ]

    def __init__(self, dim=128, seed=42):
        rng = np.random.default_rng(seed)
        self.embeddings = rng.standard_normal((len(self.PASSAGES), dim)).astype(np.float32)
        # normalise
        norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        self.embeddings /= norms
        self.dim = dim

    def embed_query(self, text: str, seed=None) -> np.ndarray:
        """Deterministic query embedding (hash-based for reproducibility)."""
        rng = np.random.default_rng(abs(hash(text)) % (2**31))
        vec = rng.standard_normal(self.dim).astype(np.float32)
        vec /= np.linalg.norm(vec)
        # bias toward first passage for demo
        vec = 0.7 * self.embeddings[0] + 0.3 * vec
        vec /= np.linalg.norm(vec)
        return vec

    def search(self, query_vec: np.ndarray, top_k: int = 3):
        scores = self.embeddings @ query_vec
        idx = np.argsort(scores)[::-1][:top_k]
        return [(self.PASSAGES[i], float(scores[i])) for i in idx]


index = DummyIndex(dim=128)
print(f'index built — {len(index.PASSAGES)} passages, dim={index.dim}')

In [ ]:
# --- run retrieval ---
QUERY = "What are the side effects of metformin?"

query_vec = index.embed_query(QUERY)
results = index.search(query_vec, top_k=3)

print(f'Query: "{QUERY}"')
print('=' * 60)
for rank, (passage, score) in enumerate(results, 1):
    print(f'\nRank {rank}  |  score={score:.4f}  |  PMID: {passage["pmid"]}')
    print(f'Section : {passage["section"]}')
    print(f'Text    : {passage["text"][:120]}...')

In [ ]:
# --- generate answer with inline citations ---

def generate_answer_with_citations(query: str, retrieved: list) -> str:
    """Template-based answer generation (no API key required for demo)."""
    context_snippets = [p['text'] for p, _ in retrieved]
    pmids = [p['pmid'] for p, _ in retrieved]

    answer = (
        f"Based on {len(retrieved)} retrieved passages, metformin's most common side effects "
        f"are gastrointestinal (nausea, vomiting, diarrhoea) [{pmids[0]}]. "
        f"A rare but serious risk is lactic acidosis, particularly in patients with renal impairment [{pmids[1]}]. "
        f"Long-term use can also cause vitamin B12 deficiency, affecting roughly 10–30% of users [{pmids[2]}]. "
        f"Unlike sulfonylureas, metformin does not directly cause hypoglycaemia."
    )
    return answer


answer = generate_answer_with_citations(QUERY, results)
print('Generated Answer')
print('=' * 60)
print(answer)

In [ ]:
# --- hallucination detection via entailment scores ---
import numpy as np

def nli_entailment_score(premise: str, hypothesis: str) -> dict:
    """
    Simulated NLI entailment scoring.
    In production this uses a cross-encoder (e.g. cross-encoder/nli-deberta-v3-base).
    """
    rng = np.random.default_rng(abs(hash(premise + hypothesis)) % (2**31))
    raw = rng.dirichlet([5, 1, 2])   # skewed toward entailment
    return {
        'entailment':   round(float(raw[0]), 3),
        'neutral':      round(float(raw[1]), 3),
        'contradiction':round(float(raw[2]), 3),
    }


# check each sentence of the generated answer against retrieved passages
claims = [
    "metformin causes gastrointestinal side effects like nausea and vomiting",
    "lactic acidosis is a rare but serious risk of metformin",
    "long-term metformin use can cause vitamin B12 deficiency",
    "metformin does not cause hypoglycaemia",
]

print(f'Hallucination Detection Report')
print('=' * 60)
for claim in claims:
    best_score = 0
    best_pmid = None
    for passage, _ in results:
        score_dict = nli_entailment_score(passage['text'], claim)
        if score_dict['entailment'] > best_score:
            best_score = score_dict['entailment']
            best_pmid = passage['pmid']

    flag = '✓ supported' if best_score > 0.5 else '⚠ unverified'
    print(f'  {flag}  (ent={best_score:.3f}, source={best_pmid})')
    print(f'    claim: "{claim[:80]}"')
    print()